On the validation data, we identify the best set of hyperparameters (epochs, train batch size, and learning rate) that minimize the loss (EMD) of the validation data predictions. 

During training, the each cell in the generated data is drawn from the same distribution as each cell in the actual data. Thus, the EMD can just be calculated across all cells. For validation, we will generate different numbers of cells for each condition (stimulation + cell type combination) as compared to the actual data. Thus, we will calculate the EMD separately on each condition and add them together. 

We run the EMD loss on in-distribution gene expression (asking the counter-factual of predicting out of distribtion conditions from in-distribution gene expression inputs). What would the TF activity of a measured cell look like in an unmeasured condition?

In [1]:
import os
import glob

import torch
import torch.nn as nn
from geomloss import SamplesLoss

import pandas as pd
import scanpy as sc
import numpy as np


[KeOps] Warning : There were warnings or errors :
<stdin>:1:10: fatal error: cuda.h: No such file or directory
compilation terminated.

[KeOps] Warning : 
    The location of Cuda header files cuda.h and nvrtc.h could not be detected on your system.
    You must determine their location and then define the environment variable CUDA_PATH,
    either before launching Python or using os.environ before importing keops. For example
    if these files are in /vol/cuda/10.2.89-cudnn7.6.4.38/include you can do :
      import os
      os.environ['CUDA_PATH'] = '/vol/cuda/10.2.89-cudnn7.6.4.38'
    
[KeOps] Compiling cuda jit compiler engine ... 
[KeOps] Warning : There were warnings or errors :
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/keopscore/binders/nvrtc/nvrtc_jit.cpp:16:10: fatal error: cuda.h: No such file or directory
 #include <cuda.h>
          ^~~~~~~~
compilation terminated.

OK
[pyKeOps] Compiling nvrtc binder for python ... 
[KeOps] W

In [2]:
import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.train import TrainSC
from scLEMBAS.preprocess import exponential_discriminator_weight

In [3]:
n_cores = 20
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')

Load the data:

In [4]:
adata = sc.read_h5ad(os.path.join(data_path, 'processed', 'kang_expr_scored.h5ad'))
sn_ppis = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_sn_ppis.csv'), index_col = 0)

tf_adata = io.read_tfad(os.path.join(data_path, 'processed', 'Kang_tf_activity.h5ad'))
tf_adata.obs['condition'] = tf_adata.obs['stim'].astype(str) + '^' + tf_adata.obs['seurat_annotations'].astype(str)

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
data_split_path = os.path.join(os.path.join(data_path, 'processed', 'data_split_barcodes'))
fold_keys = ['train_cells', 'val_cells', 'train_cond', 'val_cond']
k_fold_cells = {}
for k in range(5):
    k_fold_cells[k] = {fk: open(os.path.join(data_split_path, 'kang_' + str(k) + '_' + fk + '.txt')).read().splitlines()
                      for fk in fold_keys}

In [6]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

Setup the parameters:

In [7]:
def generate_lr_params(n_epochs, max_lr, lr_scaling_factor = 10, lr_decay = 0.9):
    lr_period = 3 if n_epochs < 500 else 4
    lr_params = {'max_epochs': n_epochs, 
                 'maximum_learning_rate': max_lr, 
                 'minimum_learning_rate': max_lr/lr_scaling_factor,
                 'lr_restart_epoch': int(n_epochs/lr_period), 
                 'reset_optimizer_epoch': int(n_epochs/3), 
                'lr_decay': lr_decay, 
                 'lr_restart_factor': 1, 
                 'warmup_epochs': int(n_epochs/10)}
    return lr_params

def generate_discriminator_params(n_epochs, max_lr, discriminator_penalty_weight, 
                                  lr_scaling_factor = 10, lr_decay = 0.9):
    general_params = generate_lr_params(n_epochs, max_lr, lr_scaling_factor = lr_scaling_factor, 
                                        lr_decay = lr_decay)
    
    keys_to_keep = ['maximum_learning_rate', 'minimum_learning_rate', 'lr_restart_epoch', 
                   'warmup_epochs', 'lr_decay', 'reset_optimizer_epoch']
    discriminator_params = {'batch_momentum': 0.01,
                            'layer_norm': False,
                            'dropout_rate': 0.1,
                            'activation_fn': nn.LeakyReLU,
                            'n_hidden_nodes': [768, 512, 256],
                            'lr_restart_factor': 1,
                            'optimizer': torch.optim.Adam,
                            'discriminator_lambda_L2': 1e-3,
                            'discriminator_penalty_weight': discriminator_penalty_weight}
    discriminator_params = {**discriminator_params, 
                           **{k:v for k,v in general_params.items() if k in keys_to_keep}}
    
    return discriminator_params

In [13]:
# vae
n_layers_vae = 2
n_nodes = len(set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()))
vae_n_hidden_nodes = list(np.round(np.linspace(adata.shape[1], n_nodes, n_layers_vae + 2)).astype(int)[1:-1])

# linear scaling of inputs/outputs
projection_amplitude_in = 1
projection_amplitude_out = 1

# other parameters
bionet_params = {'target_steps': 100, 
                 'max_steps': 120, 
                 'exp_factor':50, 
                 'tolerance': 1e-5, 
                 'leak':1e-2, 
                'cat_max_norm': 1} 
vae_params = {'vae_batch_momentum': 0.01, 'vae_layer_norm': False, 'vae_dropout_rate': 0.1,
              'vae_activation_fn': nn.LeakyReLU,
              'vae_n_hidden_nodes': vae_n_hidden_nodes, 
              'vae_var_min': 1e-4}
bionet_params = {**bionet_params, **vae_params}

# training parameters
other_params_default = {'network_noise_scale': 10, 'gradient_noise_scale': 1e-9, 
               'test_batch_size': np.nan}
spectral_radius_params = {'n_probes_spectral': 5, 
                          'power_steps_spectral': 50, 
                          'subset_n_spectral': 10}
target_spectral_radius = 0.9

regularization_params_default = {'input_lambda_L2': 0, # doesn't matter if setting the requires grad to False
                         'bn_weights_lambda_l2': 1e-7, 
                         'bn_bias_lambda_L2': 0, # don't incorporate because of KL divergence
                         'output_weights_lambda_L2': 1e-7,
                         'output_bias_lambda_L2': 1e-7,
                         'moa_lambda_L1': 1e2,  
                         'uniform_lambda_L2': 1e-7,#, 1e-5,
                         'uniform_min': 0,
                         'uniform_max': 1, 
                         'spectral_loss_factor': 1e-6,
                        'vae_lambda_l2': 1e-7, 
                        'vae_scaling_KL': 1e-2}


Iterate:

In [14]:
max_epochs_iter = [100, 300, 600, 900, 1500]
max_lr_iter = [1e-3, 1e-4]
train_batch_iter = [256, 512, 1024, 2048]

In [15]:
if os.path.isfile(os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv')):
    res = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv'), index_col = 0)
else:
    res = pd.DataFrame(columns = ['max_epochs', 'max_lr', 'train_batch_size', 'k', 'emd_loss_total', 
                                 'KL_regularization'])

stim_map = {'STIM': 1, 'CTRL': 0}
best_emd_loss = np.inf

# Start

In [16]:
res

,max_epochs,max_lr,train_batch_size,k,emd_loss_total,KL_regularization
0,100,0.001,256,0,251.878281,0.01
1,100,0.001,256,1,241.866817,0.01
2,100,0.001,256,2,253.470757,0.01
3,100,0.001,256,3,209.076370,0.01
4,100,0.001,256,4,263.759132,0.01
5,100,0.001,512,0,245.493481,0.01


In [17]:
for max_epochs in max_epochs_iter:
    
    discriminator_penalty_weight, _ = exponential_discriminator_weight(n_epochs = max_epochs,
                                                                   min_penalty_weight = 0.1,
                                                                   max_penalty_weight = 4, 
                                                                  a = 1, 
                                                                  b = 0.007)
    for max_lr in max_lr_iter:
        
        lr_params = generate_lr_params(n_epochs = max_epochs, max_lr = max_lr, lr_scaling_factor = 10, lr_decay = 0.9)
        discriminator_params = generate_discriminator_params(n_epochs = max_epochs, max_lr = max_lr, 
                              discriminator_penalty_weight = discriminator_penalty_weight, 
                              lr_scaling_factor = 10, lr_decay = 0.9)
        
        for train_batch in train_batch_iter:
            if res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) & 
                    (res.train_batch_size == train_batch)].shape[0] == 0: # no k's run
                emd_loss_k = []
                trainers_k = {}
            else:
                emd_loss_k = res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) &
                                 (res.train_batch_size == train_batch)].emd_loss_total.tolist()
                trainers_k = io.read_pickled_object(os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle'))
                
            for k in k_fold_cells:
                if res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) & 
                    (res.train_batch_size == train_batch) & (res.k == k)].shape[0] == 1:
                    run = False
                    pass
                else:
                    run = True
                    # sanity checks
                    if k in trainers_k:
                        raise ValueError('Something went wrong in loading new files')
                    elif len(trainers_k) > 0:
                        if (max(trainers_k) != k - 1):
                            raise ValueError('Something went wrong in loading new files')
                        for _k in trainers_k:
                            a = trainers_k[_k].hyper_params['max_epochs'] == max_epochs
                            b = trainers_k[_k].hyper_params['maximum_learning_rate'] == max_lr
                            c = trainers_k[_k].hyper_params['train_batch_size'] == train_batch
                            if not (a and b and c):
                                raise ValueError('Something went wrong in loading new files')    
                    break
            if run:
                break
        if run:
            break
    if run:
        break

In [13]:
# for max_epochs in max_epochs_iter:
    
#     discriminator_penalty_weight, _ = exponential_discriminator_weight(n_epochs = max_epochs,
#                                                                    min_penalty_weight = 0.1,
#                                                                    max_penalty_weight = 4, 
#                                                                   a = 1, 
#                                                                   b = 0.007)
#     for max_lr in max_lr_iter:
        
#         lr_params = generate_lr_params(n_epochs = max_epochs, max_lr = max_lr, lr_scaling_factor = 10, lr_decay = 0.9)
#         discriminator_params = generate_discriminator_params(n_epochs = max_epochs, max_lr = max_lr, 
#                               discriminator_penalty_weight = discriminator_penalty_weight, 
#                               lr_scaling_factor = 10, lr_decay = 0.9)
        
#         for train_batch in train_batch_iter:
#             if res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) & 
#                     (res.train_batch_size == train_batch)].shape[0] == 0: # no k's run
#                 emd_loss_k = []
#                 trainers_k = {}
#             else:
#                 emd_loss_k = res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) &
#                                  (res.train_batch_size == train_batch)].emd_loss_total.tolist()
#                 trainers_k = io.read_pickled_object(os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle'))
                
#             for k in k_fold_cells:
#                 if res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) & 
#                     (res.train_batch_size == train_batch) & (res.k == k)].shape[0] == 1:
#                     run = False
#                     break
#                     pass
#                 else:
#                     run = True
#                     # sanity checks
#                     if k in trainers_k:
#                         raise ValueError('Something went wrong in loading new files')
#                     elif len(trainers_k) > 0:
#                         if (max(trainers_k) != k - 1):
#                             raise ValueError('Something went wrong in loading new files')
#                         for _k in trainers_k:
#                             a = trainers_k[_k].hyper_params['max_epochs'] == max_epochs
#                             b = trainers_k[_k].hyper_params['maximum_learning_rate'] == max_lr
#                             c = trainers_k[_k].hyper_params['train_batch_size'] == train_batch
#                             if not (a and b and c):
#                                 raise ValueError('Something went wrong in loading new files')    
#                     break
#             if not run:
#                 break
#         if not run:
#             break
#     if not run:
#         break

In [18]:
print('epochs: {}, lr: {:.4f}, train batch size: {}, k: {}'.format(max_epochs, max_lr, train_batch, k))

epochs: 100, lr: 0.0010, train batch size: 512, k: 1


In [59]:
train_cells = k_fold_cells[k]['train_cells']
val_cells = k_fold_cells[k]['val_cells']

# tune other parameters as a function of those being adjusted
other_params = {**other_params_default,
                **{'train_batch_size': train_batch,
                   'validation_batch_size': len(val_cells)}}
training_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}


mod = SignalingModel(net = sn_ppis,
                     X_in = pd.DataFrame(tf_adata.obs.stim.cat.codes, columns = ['IFNB1']),
                     y_out = tf_adata.to_df().copy(), 
                     expr = adata.to_df().copy(), 
                     covariates = tf_adata.obs.copy(),
                     categorical_covariate_keys = ['seurat_annotations'],
                     projection_amplitude_in = projection_amplitude_in, 
                     projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     dtype = torch.float32, device = device, seed = seed)
# model setup
mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

# training loop
trainer = TrainSC(mod = mod,
                   prediction_optimizer = torch.optim.Adam,
                   prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device), #torch.nn.MSELoss(reduction='mean'),
                  discriminator_params = discriminator_params,
                   hyper_params = training_params,
                   train_split = {'train': train_cells, 'test': None, 'validation': val_cells}, 
                   train_seed = seed, 
                   track_test = False,
                   track_validation = False)

In [60]:
# mod = trainer.train_model(verbose = True)

In [61]:
from scLEMBAS.model.train import *
mod = mod
prediction_optimizer = torch.optim.Adam
prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device) #torch.nn.MSELoss(reduction='mean')
discriminator_params = discriminator_params
hyper_params = training_params
train_split = {'train': train_cells, 'test': None, 'validation': val_cells}
train_seed = seed,
track_test = False
track_validation = False
verbose = True
self = trainer

In [62]:
start_time = time.time()
for e in trange(self.hyper_params['max_epochs']):
    print(e)
    if e > 0:
        break
    cur_lr = self.prediction_optimizer.param_groups[0]['lr']
    self.discriminator['_cur_lr'] = self.discriminator['optimizer'].param_groups[0]['lr']
    cur_disc_lambda = self.discriminator['params']['discriminator_penalty_weight'][e]

    cur_vae_loss, cur_kl_loss, disc_loss_tot_train, disc_loss_pred_train, disc_param_loss = [], [], [], [], []

    # iterate through batches
    if self.mod.seed:
        utils.set_seeds(self.mod.seed + e)
    for batch, (X_in_, y_out_, covariates_idx_, expr_) in enumerate(self.train_dataloader):
        self.mod.train()
        for mod_discriminator in self.discriminator['discriminators'].values():
            mod_discriminator.train()

        self.prediction_optimizer.zero_grad()
        self.discriminator['optimizer'].zero_grad()

        X_in_, y_out_, covariates_idx_ = X_in_.to(self.mod.device), y_out_.to(self.mod.device), covariates_idx_.to(self.mod.device)

        # forward pass
        X_full = self.mod.input_layer(X_in_) # transform to full network with ligand input concentrations
        utils.set_seeds(self.mod.seed + self.mod._gradient_seed_counter)
        network_noise = torch.randn(X_full.shape, device = X_full.device)
        X_full = X_full + (self.hyper_params['network_noise_scale'] * cur_lr * network_noise) # randomly add noise to signaling network input, makes model more robust
        Y_full, bias_terms = self.mod.signaling_network(X_full = X_full, 
                                                         covariates_idx = covariates_idx_, 
                                                         expr = expr_) # train signaling network weights
        bias_global, bias_mu, bias_log_sigma_squared = bias_terms

        Y_hat = self.mod.output_layer(Y_full)

        ############ DISCRIMINATOR ############
        # discriminator prediction and loss
        discriminator_loss_accuracy = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
        for cat_group_idx, (cat, discriminator) in enumerate(self.discriminator['discriminators'].items()):
            bias_global_prediction = discriminator(bias_global) # predicted logits

            target = covariates_idx_[:, cat_group_idx]
            if discriminator.n_labels == 2:
                target = target.to(self.mod.dtype).unsqueeze(1)

            discriminator_loss_accuracy += discriminator.loss_fn(bias_global_prediction, target)   # if don't use retain_graph = True, then use bias_global_prediction.detach() here
#                     prediction_loss -= discriminator.loss_fn(bias_global_prediction, target) 

        # discriminator regularization
        discriminator_reg = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
        for discriminator in self.discriminator['discriminators'].values():
            discriminator_reg += discriminator.L2_reg(self.discriminator['params']['discriminator_lambda_L2'])
        discriminator_loss = discriminator_loss_accuracy + discriminator_reg

        # discriminator optimization
        # NOTE: discriminator is optimized prior to advererial training (and loss re-calculated)
        # TODO: need to reset optimizer? or anything
        discriminator_loss.backward(retain_graph = True)
        self.discriminator['optimizer'].step()

        ############ LEMBAS and generator ############
        # reconstruction loss
        prediction_loss = self.prediction_loss_fn(y_out_, Y_hat)

        # lembas regularization
        sign_reg = self.mod.signaling_network.sign_regularization(lambda_L1 = self.hyper_params['moa_lambda_L1']) # incorrect MoA
#             ligand_reg = self.mod.ligand_regularization(lambda_L2 = self.hyper_params['ligand_lambda_L2']) # ligand biases
        stability_loss, spectral_radius = self.mod.signaling_network.get_SS_loss(Y_full = Y_full.detach(), spectral_loss_factor = self.hyper_params['spectral_loss_factor'],
                                                                            subset_n = self.hyper_params['subset_n_spectral'], n_probes = self.hyper_params['n_probes_spectral'], 
                                                                            power_steps = self.hyper_params['power_steps_spectral'])
        uniform_reg = self.mod.uniform_regularization(lambda_L2 = self.hyper_params['uniform_lambda_L2']*cur_lr, Y_full = Y_full, 
                                                target_min = 0, target_max = self.hyper_params['uniform_max']) # uniform distribution
        input_param_reg, sn_param_reg, output_param_reg = self.mod.L2_reg(input_lambda_L2=self.hyper_params['input_lambda_L2'],
                                    bn_weights_lambda_l2=self.hyper_params['bn_weights_lambda_l2'], 
                                    bn_bias_lambda_L2=self.hyper_params['bn_bias_lambda_L2'], 
                                    bias_global = bias_global,
                                    output_weights_lambda_L2=self.hyper_params['output_weights_lambda_L2'],
                                    output_bias_lambda_L2=self.hyper_params['output_bias_lambda_L2'])
        param_reg = input_param_reg + sn_param_reg + output_param_reg
        vae_reg = self.mod.signaling_network.vae.L2_reg(lambda_L2=self.hyper_params['vae_lambda_l2']) # VAE loss
        param_reg += vae_reg

        # NOTE: KL divergence is scaled to match loss magnitudes; no bias regularization given KL regularization
        # can use MMD in the future if KL unstable
        kl_divergence = self.mod.signaling_network.vae.KL_divergence(z_mu = bias_mu, 
                                                                     z_log_sigma_squared = bias_log_sigma_squared, 
                                                                     scaling_factor = self.hyper_params['vae_scaling_KL'])

        tot_pred_loss = prediction_loss + sign_reg + param_reg + stability_loss + uniform_reg + kl_divergence

        # adverserial portion -- same as discriminator, but recalculating on trained model
        # TODO: does discriminator need to be in .eval/.inference mode?
        adverserial_loss = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
        for cat_group_idx, (cat, discriminator) in enumerate(self.discriminator['discriminators'].items()):
            bias_global_prediction = discriminator(bias_global) # predicted logits

            target = covariates_idx_[:, cat_group_idx]
            if discriminator.n_labels == 2:
                target = target.to(self.mod.dtype).unsqueeze(1)

            adverserial_loss += discriminator.loss_fn(bias_global_prediction, target)   # if don't use retain_graph = True, then use bias_global_prediction.detach() here
#                     prediction_loss -= discriminator.loss_fn(bias_global_prediction, target) 
        tot_pred_loss -= (cur_disc_lambda*adverserial_loss) # adversarial portion

        # gradient
        tot_pred_loss.backward()
        self.mod.add_gradient_noise(noise_level = self.hyper_params['gradient_noise_scale'])
        self.prediction_optimizer.step()


        sv = np.array([e + 1, batch, cur_lr, self.discriminator['_cur_lr'], 
                       time.time() - start_time, spectral_radius, 
            self.mod.signaling_network.count_sign_mismatch(), 
            tot_pred_loss.item(), prediction_loss.item(), #train_pearson_r, 
            sign_reg.item(), stability_loss.item(), uniform_reg.item(), 
            input_param_reg.item(), sn_param_reg.item(), output_param_reg.item(),
            vae_reg.item(), kl_divergence.item(), 
                       cur_disc_lambda*adverserial_loss.item(), discriminator_loss.item(), discriminator_loss_accuracy.item(), discriminator_reg.item()])
        self.stats['train'] = np.vstack((self.stats['train'], sv))

        # free up CUDA mem
        del sign_reg, stability_loss, uniform_reg, param_reg, prediction_loss
        del input_param_reg, sn_param_reg, output_param_reg
        del vae_reg, kl_divergence, discriminator_loss, discriminator_loss_accuracy, discriminator_reg
        del X_in_, y_out_, covariates_idx_, X_full, Y_full, Y_hat
        torch.cuda.empty_cache()

    self.lr_scheduler.step()
    self.discriminator['lr_scheduler'].step()

    # test/validation
    if self.track_validation or self.track_test:
        self.mod.eval()
        with torch.inference_mode(): 
            if self.track_validation:
                # loss_val_all = []
                # pearson_val_all = []
                for batch, (X_in_val, y_out_val, covariates_idx_val, expr_val) in enumerate(self.validation_dataloader): 
                    X_in_val, y_out_val, covariates_idx_val = X_in_val.to(self.mod.device), y_out_val.to(self.mod.device), covariates_idx_val.to(self.mod.device)
                    self.mod.signaling_network.mask = self.mod.signaling_network.mask.to(X_in_val.device)
                    y_pred_val, _, _ = self.mod(X_in = X_in_val, covariates_idx = covariates_idx_val, expr = expr_val)
                    loss_val = self.prediction_loss_fn(y_out_val, y_pred_val).detach().item()
                    # loss_val_all.append(loss_val)
                    # pearson_val_all.append(pearson_val)
                    self.stats['validation'] = np.vstack((self.stats['validation'], np.array([e+1, batch, loss_val])))
                    del y_pred_val, _
            if self.track_test:
                # loss_test_all = []
                # pearson_test_all = []
                for batch, (X_in_test, y_out_test, covariates_idx_test, expr_test) in enumerate(self.test_dataloader):
                    X_in_test, y_out_test, covariates_idx_test = X_in_test.to(self.mod.device), y_out_test.to(self.mod.device), covariates_idx_test.to(self.mod.device)
                    y_pred_test, _, _ = self.mod(X_in = X_in_test, covariates_idx = covariates_idx_test, expr = expr_test)
                    loss_test = self.prediction_loss_fn(y_out_test, y_pred_test).detach().item()
                    # loss_test_all.append(loss_test)
                    self.stats['test'] = np.vstack((self.stats['test'], np.array([e +1, batch, loss_test])))
                    del y_pred_test, _

    if e % (self.hyper_params['max_epochs']/100) == 0:
        param_names = []
        for name, param in self.mod.named_parameters():
            if torch.isnan(param).any():
                param_names.append(name)
        if len(param_names) > 0:
            log_error = 'NaN values found in model parameters at epoch {}'.format(e)
            log_error += ' for layers ' + ', '.join(param_names)
            logging.error(log_error)
            raise ValueError(log_error)
        if verbose:
            self.print_stats(e)

    if np.logical_and(e % self.hyper_params['reset_optimizer_epoch'] == 0, e>0):
        self.prediction_optimizer.state = self.reset_state.copy()
    if (e % self.discriminator['params']['reset_optimizer_epoch'] == 0) and e > 0:
        self.discriminator['optimizer'].state = self.reset_state.copy()

  0%|                                                     | 0/100 [00:00<?, ?it/s]

0


  1%|▍                                            | 1/100 [00:09<15:56,  9.66s/it]

i=0, l(tr)=70.69336, s=0.23647, r=0.00010, v=0.00000
1


In [63]:
e

1

In [64]:
cur_lr = self.prediction_optimizer.param_groups[0]['lr']
self.discriminator['_cur_lr'] = self.discriminator['optimizer'].param_groups[0]['lr']
cur_disc_lambda = self.discriminator['params']['discriminator_penalty_weight'][e]

cur_vae_loss, cur_kl_loss, disc_loss_tot_train, disc_loss_pred_train, disc_param_loss = [], [], [], [], []

# iterate through batches
if self.mod.seed:
    utils.set_seeds(self.mod.seed + e)

In [65]:
batch_idx = 6
weights_list = []

for batch, (X_in_, y_out_, covariates_idx_, expr_) in enumerate(self.train_dataloader):
    print(batch)
    if batch > batch_idx:
        break
    self.mod.train()
    for mod_discriminator in self.discriminator['discriminators'].values():
        mod_discriminator.train()

    self.prediction_optimizer.zero_grad()
    self.discriminator['optimizer'].zero_grad()

    X_in_, y_out_, covariates_idx_ = X_in_.to(self.mod.device), y_out_.to(self.mod.device), covariates_idx_.to(self.mod.device)

    # forward pass
    X_full = self.mod.input_layer(X_in_) # transform to full network with ligand input concentrations
    utils.set_seeds(self.mod.seed + self.mod._gradient_seed_counter)
    network_noise = torch.randn(X_full.shape, device = X_full.device)
    X_full = X_full + (self.hyper_params['network_noise_scale'] * cur_lr * network_noise) # randomly add noise to signaling network input, makes model more robust
    Y_full, bias_terms = self.mod.signaling_network(X_full = X_full, 
                                                     covariates_idx = covariates_idx_, 
                                                     expr = expr_) # train signaling network weights
    bias_global, bias_mu, bias_log_sigma_squared = bias_terms

    Y_hat = self.mod.output_layer(Y_full)

    ############ DISCRIMINATOR ############
    # discriminator prediction and loss
    discriminator_loss_accuracy = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
    for cat_group_idx, (cat, discriminator) in enumerate(self.discriminator['discriminators'].items()):
        bias_global_prediction = discriminator(bias_global) # predicted logits

        target = covariates_idx_[:, cat_group_idx]
        if discriminator.n_labels == 2:
            target = target.to(self.mod.dtype).unsqueeze(1)

        discriminator_loss_accuracy += discriminator.loss_fn(bias_global_prediction, target)   # if don't use retain_graph = True, then use bias_global_prediction.detach() here
#                     prediction_loss -= discriminator.loss_fn(bias_global_prediction, target) 

    # discriminator regularization
    discriminator_reg = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
    for discriminator in self.discriminator['discriminators'].values():
        discriminator_reg += discriminator.L2_reg(self.discriminator['params']['discriminator_lambda_L2'])
    discriminator_loss = discriminator_loss_accuracy + discriminator_reg

    # discriminator optimization
    # NOTE: discriminator is optimized prior to advererial training (and loss re-calculated)
    # TODO: need to reset optimizer? or anything
    discriminator_loss.backward(retain_graph = True)
    self.discriminator['optimizer'].step()

    ############ LEMBAS and generator ############
    # reconstruction loss
    prediction_loss = self.prediction_loss_fn(y_out_, Y_hat)

    # lembas regularization
    sign_reg = self.mod.signaling_network.sign_regularization(lambda_L1 = self.hyper_params['moa_lambda_L1']) # incorrect MoA
#             ligand_reg = self.mod.ligand_regularization(lambda_L2 = self.hyper_params['ligand_lambda_L2']) # ligand biases
    stability_loss, spectral_radius = self.mod.signaling_network.get_SS_loss(Y_full = Y_full.detach(), spectral_loss_factor = self.hyper_params['spectral_loss_factor'],
                                                                        subset_n = self.hyper_params['subset_n_spectral'], n_probes = self.hyper_params['n_probes_spectral'], 
                                                                        power_steps = self.hyper_params['power_steps_spectral'])
    uniform_reg = self.mod.uniform_regularization(lambda_L2 = self.hyper_params['uniform_lambda_L2']*cur_lr, Y_full = Y_full, 
                                            target_min = 0, target_max = self.hyper_params['uniform_max']) # uniform distribution
    input_param_reg, sn_param_reg, output_param_reg = self.mod.L2_reg(input_lambda_L2=self.hyper_params['input_lambda_L2'],
                                bn_weights_lambda_l2=self.hyper_params['bn_weights_lambda_l2'], 
                                bn_bias_lambda_L2=self.hyper_params['bn_bias_lambda_L2'], 
                                bias_global = bias_global,
                                output_weights_lambda_L2=self.hyper_params['output_weights_lambda_L2'],
                                output_bias_lambda_L2=self.hyper_params['output_bias_lambda_L2'])
    param_reg = input_param_reg + sn_param_reg + output_param_reg
    vae_reg = self.mod.signaling_network.vae.L2_reg(lambda_L2=self.hyper_params['vae_lambda_l2']) # VAE loss
    param_reg += vae_reg

    # NOTE: KL divergence is scaled to match loss magnitudes; no bias regularization given KL regularization
    # can use MMD in the future if KL unstable
    kl_divergence = self.mod.signaling_network.vae.KL_divergence(z_mu = bias_mu, 
                                                                 z_log_sigma_squared = bias_log_sigma_squared, 
                                                                 scaling_factor = self.hyper_params['vae_scaling_KL'])

    tot_pred_loss = prediction_loss + sign_reg + param_reg + stability_loss + uniform_reg + kl_divergence

    # adverserial portion -- same as discriminator, but recalculating on trained model
    # TODO: does discriminator need to be in .eval/.inference mode?
    adverserial_loss = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
    for cat_group_idx, (cat, discriminator) in enumerate(self.discriminator['discriminators'].items()):
        bias_global_prediction = discriminator(bias_global) # predicted logits

        target = covariates_idx_[:, cat_group_idx]
        if discriminator.n_labels == 2:
            target = target.to(self.mod.dtype).unsqueeze(1)

        adverserial_loss += discriminator.loss_fn(bias_global_prediction, target)   # if don't use retain_graph = True, then use bias_global_prediction.detach() here
#                     prediction_loss -= discriminator.loss_fn(bias_global_prediction, target) 
    tot_pred_loss -= (cur_disc_lambda*adverserial_loss) # adversarial portion

    # gradient
    tot_pred_loss.backward()
    self.mod.add_gradient_noise(noise_level = self.hyper_params['gradient_noise_scale'])
    self.prediction_optimizer.step()


    sv = np.array([e + 1, batch, cur_lr, self.discriminator['_cur_lr'], 
                   time.time() - start_time, spectral_radius, 
        self.mod.signaling_network.count_sign_mismatch(), 
        tot_pred_loss.item(), prediction_loss.item(), #train_pearson_r, 
        sign_reg.item(), stability_loss.item(), uniform_reg.item(), 
        input_param_reg.item(), sn_param_reg.item(), output_param_reg.item(),
        vae_reg.item(), kl_divergence.item(), 
                   cur_disc_lambda*adverserial_loss.item(), discriminator_loss.item(), discriminator_loss_accuracy.item(), discriminator_reg.item()])
    self.stats['train'] = np.vstack((self.stats['train'], sv))

    # free up CUDA mem
    del sign_reg, stability_loss, uniform_reg, param_reg, prediction_loss
    del input_param_reg, sn_param_reg, output_param_reg
    del vae_reg, kl_divergence, discriminator_loss, discriminator_loss_accuracy, discriminator_reg
    
    for name, param in mod.signaling_network.named_parameters():
        if name == 'weights':
            weights_list.append(param.detach().cpu().numpy())
    
    del X_in_, y_out_, covariates_idx_, X_full, Y_full, Y_hat
        
        
        
    torch.cuda.empty_cache()

0
1
2
3
4
5
6
7


In [66]:
weights_list[0]

array([[-2.0275431e-05, -3.9266935e-05, -2.9351459e-05, ...,
        -2.7870239e-05, -5.3026284e-05, -2.2114085e-05],
       [ 4.4769193e-05,  4.0196806e-05,  3.8968265e-05, ...,
         4.0315270e-05,  3.9429386e-05,  3.2794313e-05],
       [ 6.4799940e-05,  7.7204953e-05,  8.1216269e-05, ...,
         6.2503939e-05,  7.0670685e-05,  8.6581771e-05],
       ...,
       [-1.6952516e-05, -1.9450399e-05,  1.8163333e-05, ...,
        -5.6741581e-05, -1.6742062e-05, -2.4676026e-05],
       [-4.2955085e-06, -4.5665256e-05, -1.8650473e-05, ...,
        -3.1818105e-05, -5.4802469e-05, -4.1021085e-05],
       [ 8.6974520e-05,  5.7698442e-05,  4.1066174e-05, ...,
         8.1619844e-05,  8.4112115e-05,  8.5116975e-05]], dtype=float32)

In [68]:
weights_list[-2]

array([[-2.4596397e-05, -4.6790170e-05, -3.5971258e-05, ...,
        -2.9951985e-05, -5.7687677e-05, -3.3783828e-05],
       [ 5.3888103e-05,  4.5564691e-05,  5.0251609e-05, ...,
         4.8120812e-05,  4.9196595e-05,  4.3287775e-05],
       [ 4.2536369e-05,  1.2977260e-05,  4.7473171e-05, ...,
         3.5355511e-05,  4.4126562e-05,  3.6838588e-05],
       ...,
       [-4.0497807e-06,  4.4207391e-06,  4.8730857e-05, ...,
         5.3333783e-06,  1.6765978e-05,  5.2952557e-05],
       [ 4.0188184e-05, -7.6667607e-07,  4.5812805e-05, ...,
         2.0029907e-05, -7.8334870e-06, -5.5446794e-06],
       [ 5.4479500e-05,  3.2687956e-05,  2.0885876e-05, ...,
         6.7443631e-05,  4.6866782e-05,  6.3039290e-05]], dtype=float32)

In [57]:
# # for param in mod.signaling_network.parameters():
# #     print(param)
# for name, param in mod.signaling_network.named_parameters():
#     print(f"Parameter name: {name}")
#     print(param)

In [23]:
batch

7

In [ ]:
# you are here

In [23]:
self.mod.train()
for mod_discriminator in self.discriminator['discriminators'].values():
    mod_discriminator.train()

self.prediction_optimizer.zero_grad()
self.discriminator['optimizer'].zero_grad()

X_in_, y_out_, covariates_idx_ = X_in_.to(self.mod.device), y_out_.to(self.mod.device), covariates_idx_.to(self.mod.device)

# forward pass
X_full = self.mod.input_layer(X_in_) # transform to full network with ligand input concentrations
utils.set_seeds(self.mod.seed + self.mod._gradient_seed_counter)
network_noise = torch.randn(X_full.shape, device = X_full.device)
X_full = X_full + (self.hyper_params['network_noise_scale'] * cur_lr * network_noise) # randomly add noise to signaling network input, makes model more robust
Y_full, bias_terms = self.mod.signaling_network(X_full = X_full, 
                                                 covariates_idx = covariates_idx_, 
                                                 expr = expr_) # train signaling network weights
bias_global, bias_mu, bias_log_sigma_squared = bias_terms

Y_hat = self.mod.output_layer(Y_full)

############ DISCRIMINATOR ############
# discriminator prediction and loss
discriminator_loss_accuracy = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
for cat_group_idx, (cat, discriminator) in enumerate(self.discriminator['discriminators'].items()):
    bias_global_prediction = discriminator(bias_global) # predicted logits

    target = covariates_idx_[:, cat_group_idx]
    if discriminator.n_labels == 2:
        target = target.to(self.mod.dtype).unsqueeze(1)

    discriminator_loss_accuracy += discriminator.loss_fn(bias_global_prediction, target)   # if don't use retain_graph = True, then use bias_global_prediction.detach() here
#                     prediction_loss -= discriminator.loss_fn(bias_global_prediction, target) 

# discriminator regularization
discriminator_reg = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
for discriminator in self.discriminator['discriminators'].values():
    discriminator_reg += discriminator.L2_reg(self.discriminator['params']['discriminator_lambda_L2'])
discriminator_loss = discriminator_loss_accuracy + discriminator_reg

# discriminator optimization
# NOTE: discriminator is optimized prior to advererial training (and loss re-calculated)
# TODO: need to reset optimizer? or anything
discriminator_loss.backward(retain_graph = True)
self.discriminator['optimizer'].step()

############ LEMBAS and generator ############
# reconstruction loss
prediction_loss = self.prediction_loss_fn(y_out_, Y_hat)

# lembas regularization
sign_reg = self.mod.signaling_network.sign_regularization(lambda_L1 = self.hyper_params['moa_lambda_L1']) # incorrect MoA
#             ligand_reg = self.mod.ligand_regularization(lambda_L2 = self.hyper_params['ligand_lambda_L2']) # ligand biases
stability_loss, spectral_radius = self.mod.signaling_network.get_SS_loss(Y_full = Y_full.detach(), spectral_loss_factor = self.hyper_params['spectral_loss_factor'],
                                                                    subset_n = self.hyper_params['subset_n_spectral'], n_probes = self.hyper_params['n_probes_spectral'], 
                                                                    power_steps = self.hyper_params['power_steps_spectral'])
uniform_reg = self.mod.uniform_regularization(lambda_L2 = self.hyper_params['uniform_lambda_L2']*cur_lr, Y_full = Y_full, 
                                        target_min = 0, target_max = self.hyper_params['uniform_max']) # uniform distribution
input_param_reg, sn_param_reg, output_param_reg = self.mod.L2_reg(input_lambda_L2=self.hyper_params['input_lambda_L2'],
                            bn_weights_lambda_l2=self.hyper_params['bn_weights_lambda_l2'], 
                            bn_bias_lambda_L2=self.hyper_params['bn_bias_lambda_L2'], 
                            bias_global = bias_global,
                            output_weights_lambda_L2=self.hyper_params['output_weights_lambda_L2'],
                            output_bias_lambda_L2=self.hyper_params['output_bias_lambda_L2'])
param_reg = input_param_reg + sn_param_reg + output_param_reg
vae_reg = self.mod.signaling_network.vae.L2_reg(lambda_L2=self.hyper_params['vae_lambda_l2']) # VAE loss
param_reg += vae_reg

# NOTE: KL divergence is scaled to match loss magnitudes; no bias regularization given KL regularization
# can use MMD in the future if KL unstable
kl_divergence = self.mod.signaling_network.vae.KL_divergence(z_mu = bias_mu, 
                                                             z_log_sigma_squared = bias_log_sigma_squared, 
                                                             scaling_factor = self.hyper_params['vae_scaling_KL'])

tot_pred_loss = prediction_loss + sign_reg + param_reg + stability_loss + uniform_reg + kl_divergence

# adverserial portion -- same as discriminator, but recalculating on trained model
# TODO: does discriminator need to be in .eval/.inference mode?
adverserial_loss = torch.tensor(0, device = self.mod.device, dtype = self.mod.dtype)
for cat_group_idx, (cat, discriminator) in enumerate(self.discriminator['discriminators'].items()):
    bias_global_prediction = discriminator(bias_global) # predicted logits

    target = covariates_idx_[:, cat_group_idx]
    if discriminator.n_labels == 2:
        target = target.to(self.mod.dtype).unsqueeze(1)

    adverserial_loss += discriminator.loss_fn(bias_global_prediction, target)   # if don't use retain_graph = True, then use bias_global_prediction.detach() here
#                     prediction_loss -= discriminator.loss_fn(bias_global_prediction, target) 
tot_pred_loss -= (cur_disc_lambda*adverserial_loss) # adversarial portion

# gradient
tot_pred_loss.backward()
self.mod.add_gradient_noise(noise_level = self.hyper_params['gradient_noise_scale'])
self.prediction_optimizer.step()


sv = np.array([e + 1, batch, cur_lr, self.discriminator['_cur_lr'], 
               time.time() - start_time, spectral_radius, 
    self.mod.signaling_network.count_sign_mismatch(), 
    tot_pred_loss.item(), prediction_loss.item(), #train_pearson_r, 
    sign_reg.item(), stability_loss.item(), uniform_reg.item(), 
    input_param_reg.item(), sn_param_reg.item(), output_param_reg.item(),
    vae_reg.item(), kl_divergence.item(), 
               cur_disc_lambda*adverserial_loss.item(), discriminator_loss.item(), discriminator_loss_accuracy.item(), discriminator_reg.item()])
self.stats['train'] = np.vstack((self.stats['train'], sv))

# free up CUDA mem
del sign_reg, stability_loss, uniform_reg, param_reg, prediction_loss
del input_param_reg, sn_param_reg, output_param_reg
del vae_reg, kl_divergence, discriminator_loss, discriminator_loss_accuracy, discriminator_reg
del X_in_, y_out_, covariates_idx_, X_full, Y_full, Y_hat
torch.cuda.empty_cache()

ValueError: arange: cannot compute length

In [25]:
y_out_

tensor([[-0.0401,  0.5356,  0.3533,  ...,  0.1187, -0.2098, -0.4649],
        [-0.3015,  0.0371,  0.1691,  ..., -0.8116, -2.4844, -0.2256],
        [-0.0805, -0.1416,  0.5162,  ...,  0.2990,  0.0802, -0.3689],
        ...,
        [ 0.1216,  0.1929, -0.2301,  ..., -0.7009,  0.1030, -0.1417],
        [ 0.0150,  1.1343,  0.6851,  ..., -0.4330, -0.2188,  0.3841],
        [-0.3035,  0.4481,  1.2471,  ..., -0.1939,  0.1085, -0.2359]],
       device='cuda:0')

In [26]:
Y_hat

tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<AddBackward0>)

In [25]:
train_cells = k_fold_cells[k]['train_cells']
val_cells = k_fold_cells[k]['val_cells']

# tune other parameters as a function of those being adjusted
other_params = {**other_params_default,
                **{'train_batch_size': train_batch,
                   'validation_batch_size': len(val_cells)}}

In [26]:
regularization_params = regularization_params_default.copy()

In [27]:
regularization_params['vae_scaling_KL'] *= 2

In [28]:
# training_params['maximum_learning_rate'] /= 10
# training_params['minimum_learning_rate'] /= 10

In [ ]:
training_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}

mod = SignalingModel(net = sn_ppis,
                     X_in = pd.DataFrame(tf_adata.obs.stim.cat.codes, columns = ['IFNB1']),
                     y_out = tf_adata.to_df().copy(), 
                     expr = adata.to_df().copy(), 
                     covariates = tf_adata.obs.copy(),
                     categorical_covariate_keys = ['seurat_annotations'],
                     projection_amplitude_in = projection_amplitude_in, 
                     projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     dtype = torch.float32, device = device, seed = seed)
# model setup
mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

# training loop
trainer = TrainSC(mod = mod,
                   prediction_optimizer = torch.optim.Adam,
                   prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device), #torch.nn.MSELoss(reduction='mean'),
                  discriminator_params = discriminator_params,
                   hyper_params = training_params,
                   train_split = {'train': train_cells, 'test': None, 'validation': val_cells}, 
                   train_seed = seed, 
                   track_test = False,
                   track_validation = False)
mod = trainer.train_model(verbose = False)

  1%|▍                                            | 1/100 [00:09<16:08,  9.78s/it]

In [36]:
trainer.hyper_params

{'max_epochs': 100,
 'maximum_learning_rate': 0.001,
 'minimum_learning_rate': 0.0001,
 'lr_restart_epoch': 33,
 'reset_optimizer_epoch': 33,
 'lr_decay': 0.9,
 'lr_restart_factor': 1,
 'warmup_epochs': 10,
 'train_batch_size': 512,
 'test_batch_size': nan,
 'validation_batch_size': 2008,
 'network_noise_scale': 10,
 'gradient_noise_scale': 1e-09,
 'input_lambda_L2': 0,
 'bn_weights_lambda_l2': 1e-07,
 'bn_bias_lambda_L2': 0,
 'output_weights_lambda_L2': 1e-07,
 'output_bias_lambda_L2': 1e-07,
 'moa_lambda_L1': 100.0,
 'uniform_lambda_L2': 1e-07,
 'uniform_min': 0,
 'uniform_max': 1,
 'spectral_loss_factor': 1e-06,
 'vae_lambda_l2': 1e-07,
 'vae_scaling_KL': 0.01,
 'n_probes_spectral': 5,
 'power_steps_spectral': 50,
 'subset_n_spectral': 10}

In [93]:
res['KL_regularization'] = regularization_params_default['vae_scaling_KL']

In [95]:
regularization_params_default['vae_scaling_KL']

0.01

# End

In [ ]:
max_err_iter = 3

In [52]:
for max_epochs in max_epochs_iter:
    
    discriminator_penalty_weight, _ = exponential_discriminator_weight(n_epochs = max_epochs,
                                                                   min_penalty_weight = 0.1,
                                                                   max_penalty_weight = 4, 
                                                                  a = 1, 
                                                                  b = 0.007)
    for max_lr in max_lr_iter:
        
        lr_params = generate_lr_params(n_epochs = max_epochs, max_lr = max_lr, lr_scaling_factor = 10, lr_decay = 0.9)
        discriminator_params = generate_discriminator_params(n_epochs = max_epochs, max_lr = max_lr, 
                              discriminator_penalty_weight = discriminator_penalty_weight, 
                              lr_scaling_factor = 10, lr_decay = 0.9)
        
        for train_batch in train_batch_iter:
            if res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) & 
                    (res.train_batch_size == train_batch)].shape[0] == 0: # no k's run
                emd_loss_k = []
                trainers_k = {}
            else:
                emd_loss_k = res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) &
                                 (res.train_batch_size == train_batch)].emd_loss_total.tolist()
                trainers_k = io.read_pickled_object(os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle'))
                
            for k in k_fold_cells:
                if res[(res.max_epochs == max_epochs) & (res.max_lr == max_lr) & 
                    (res.train_batch_size == train_batch) & (res.k == k)].shape[0] == 1:
                    run = False
                    pass
                else:
                    run = True
                    # sanity checks
                    if k in trainers_k:
                        raise ValueError('Something went wrong in loading new files')
                    elif len(trainers_k) > 0:
                        if (max(trainers_k) != k - 1):
                            raise ValueError('Something went wrong in loading new files')
                        for _k in trainers_k:
                            if not np.isnan(trainers_k[_k]):
                                a = trainers_k[_k].hyper_params['max_epochs'] == max_epochs
                                b = trainers_k[_k].hyper_params['maximum_learning_rate'] == max_lr
                                c = trainers_k[_k].hyper_params['train_batch_size'] == train_batch
                                if not (a and b and c):
                                    raise ValueError('Something went wrong in loading new files')    
                        
                        
                    print('epochs: {}, lr: {:.4f}, train batch size: {}, k: {}'.format(max_epochs, max_lr, train_batch, k))
                    print('------')
                    
                    train_cells = k_fold_cells[k]['train_cells']
                    val_cells = k_fold_cells[k]['val_cells']

                    # tune other parameters as a function of those being adjusted
                    regularization_params = regularization_params_default.copy()
                    other_params = {**other_params_default,
                                    **{'train_batch_size': train_batch,
                                       'validation_batch_size': len(val_cells)}}
                    training_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}


                    mod = SignalingModel(net = sn_ppis,
                                         X_in = pd.DataFrame(tf_adata.obs.stim.cat.codes, columns = ['IFNB1']),
                                         y_out = tf_adata.to_df().copy(), 
                                         expr = adata.to_df().copy(), 
                                         covariates = tf_adata.obs.copy(),
                                         categorical_covariate_keys = ['seurat_annotations'],
                                         projection_amplitude_in = projection_amplitude_in, 
                                         projection_amplitude_out = projection_amplitude_out,
                                         weight_label = weight_label, source_label = source_label, target_label = target_label,
                                         bionet_params = bionet_params, 
                                         dtype = torch.float32, device = device, seed = seed)
                    # model setup
                    mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
                    mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

                    # training loop
                    trainer = TrainSC(mod = mod,
                                       prediction_optimizer = torch.optim.Adam,
                                       prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device), #torch.nn.MSELoss(reduction='mean'),
                                      discriminator_params = discriminator_params,
                                       hyper_params = training_params,
                                       train_split = {'train': train_cells, 'test': None, 'validation': val_cells}, 
                                       train_seed = seed, 
                                       track_test = False,
                                       track_validation = False)
                    try:
                        mod = trainer.train_model(verbose = False)
                    except:
                        err = True
                        err_iter = 0
                        while err and (err_iter < max_err_iter): 
                            try: # try with higher regularization of KL which prevents NaN errors
                                regularization_params['vae_scaling_KL'] *= 10
                                training_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}

                                mod = SignalingModel(net = sn_ppis,
                                                     X_in = pd.DataFrame(tf_adata.obs.stim.cat.codes, columns = ['IFNB1']),
                                                     y_out = tf_adata.to_df().copy(), 
                                                     expr = adata.to_df().copy(), 
                                                     covariates = tf_adata.obs.copy(),
                                                     categorical_covariate_keys = ['seurat_annotations'],
                                                     projection_amplitude_in = projection_amplitude_in, 
                                                     projection_amplitude_out = projection_amplitude_out,
                                                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                                                     bionet_params = bionet_params, 
                                                     dtype = torch.float32, device = device, seed = seed)
                                # model setup
                                mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
                                mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

                                # training loop
                                trainer = TrainSC(mod = mod,
                                                   prediction_optimizer = torch.optim.Adam,
                                                   prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device), #torch.nn.MSELoss(reduction='mean'),
                                                  discriminator_params = discriminator_params,
                                                   hyper_params = training_params,
                                                   train_split = {'train': train_cells, 'test': None, 'validation': val_cells}, 
                                                   train_seed = seed, 
                                                   track_test = False,
                                                   track_validation = False)
                                mod = trainer.train_model(verbose = False)
                                err = False
                            except: 
                                err_iter += 1
                    if err:
                        emd_loss_tot = np.nan 
                        torch.cuda.empty_cache()
                    else:        
                        
                        # prediction on in-distribution gene expression inputs, per condition
                        emd_loss_tot = 0
                        for cond in k_fold_cells[k]['val_cond']:
                            loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
                            stim, ct = cond.split('^')

                            vall_cells_cond = tf_adata.obs[(tf_adata.obs.index.isin(val_cells)) & (tf_adata.obs['condition'] == cond)].index.tolist()
                            y_val = mod.df_to_tensor(trainer.y_validation.loc[vall_cells_cond, :])

                            # in distribution gene expression!
                            expr_val = mod.df_to_tensor(mod.expr.loc[train_cells, :])

                            # set the stimulation condition we want to predict
                            X_val = pd.DataFrame(data = {'IFNB1': [stim_map[stim]]*len(train_cells)})
                            X_val = mod.df_to_tensor(X_val)

                            # set the cell type we want to predict
                            cov_idx_map = dict(zip(mod.signaling_network.covariates['seurat_annotations'], 
                                                   mod.signaling_network.covariates_idx['seurat_annotations']))
                            covariates_idx_val = torch.tensor([cov_idx_map[ct]]*len(train_cells), device = mod.device, dtype = torch.int64).view(-1,1)


                            mod.eval()
                            with torch.inference_mode():
                                y_predicted, _, _ = mod(X_in = X_val, covariates_idx = covariates_idx_val, expr = expr_val)

                            emd_loss_tot += loss_fn(y_predicted, y_val).detach().cpu().item()
                            del expr_val, X_val, covariates_idx_val, y_predicted, loss_fn
                            torch.cuda.empty_cache()

                    emd_loss_k.append(emd_loss_tot)
                    res.loc[res.shape[0], :] = [max_epochs, max_lr, train_batch, k, emd_loss_tot, 
                                               regularization_params['vae_scaling_KL']]
                    
                    trainers_k[k] = trainer
                    io.write_pickled_object(trainers_k, os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle'))
                    res.to_csv(os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv'))
                    
            emd_loss_mean = np.nan(emd_loss_k)
            if emd_loss_mean < best_emd_loss:
                best_emd_loss = emd_loss_mean
                for k, trainer in trainers_k.items():
                    io.write_pickled_object(trainer, os.path.join(models_path, 'Kang_best_trainer_' + str(k) + '.pickle'))
            if run:
                del trainers_k, trainer, mod
                torch.cuda.empty_cache()

In [53]:
files = [os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv'), 
         os.path.join(models_path, 'Kang_best_trainer_*'), 
         os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle')]
for i, f in enumerate(files):
    print('rm ' + f)
    if i == len(files) - 1:
        print('')
        print('')

rm /nobackup/users/hmbaghda/scLEMBAS/analysis/processed/Kang_k_fold_validation_results.csv
rm /nobackup/users/hmbaghda/scLEMBAS/analysis/processed/models/Kang_best_trainer_*
rm /nobackup/users/hmbaghda/scLEMBAS/analysis/processed/Kang_trainers_k.pickle




In [6]:
# test = io.read_pickled_object(os.path.join(data_path, 'processed', 'full_run_Kang_trainers_k.pickle'))
# res = pd.read_csv(os.path.join(data_path, 'processed', 'full_run_Kang_k_fold_validation_results.csv'), index_col = 0)

